<a href="https://colab.research.google.com/github/maximusrome/financial-sentiment-comparison/blob/main/train_logreg_tfidf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from data_loader import build_and_save_split

build_and_save_split()

Split already exists at /content/financial-sentiment-comparison/data/splits.parquet. Pass force=True to regenerate.


,sentence_id,text,label,agreement_tier,split
0,s00000,"According to Gran , the company has no plans t...",1,100,test
1,s00001,Technopolis plans to develop in stages an area...,1,66-74,train
2,s00002,The international electronic industry company ...,0,50-65,train
3,s00003,With the new production plant the company woul...,2,75-99,train
4,s00004,According to the company 's updated strategy f...,2,66-74,train
...,...,...,...,...,...
4833,s04833,LONDON MarketWatch -- Share prices ended lower...,0,100,val
4834,s04834,Rinkuskiai 's beer sales fell by 6.5 per cent ...,1,66-74,train
4835,s04835,Operating profit fell to EUR 35.4 mn from EUR ...,0,100,test
4836,s04836,Net sales of the Paper segment decreased to EU...,0,50-65,test


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from evaluation import evaluate_predictions
from data_loader import load_split
from pathlib import Path


REPO_ROOT = Path(__file__).resolve().parent
LM_PATH = REPO_ROOT / "data" / "Loughran-McDonald_MasterDictionary_1993-2024.csv"

train_df, val_df, test_df = load_split()

train_texts = train_df["text"].tolist()
val_texts   = val_df["text"].tolist()
test_texts  = test_df["text"].tolist()

y_train = train_df["label"].tolist()
y_val   = val_df["label"].tolist()
y_test  = test_df["label"].tolist()


all_texts = train_texts + val_texts + test_texts
tokenized_texts = [text.lower().split() for text in all_texts]

D = len(all_texts)


Vocab = set()
for text in tokenized_texts:
    Vocab.update(text)

Vocab = sorted(list(Vocab))
wordidx = {word: i for i, word in enumerate(Vocab)}
V = len(Vocab)

tfmatrix = np.zeros((V, D))

for i, tokens in enumerate(tokenized_texts):
    doc_len = len(tokens)
    if doc_len == 0:
        continue
    for word in tokens:
        if word in wordidx:
            idx = wordidx[word]
            tfmatrix[idx, i] += 1 / doc_len


dfmatrix = np.zeros(V)

for i, word in enumerate(Vocab):
    for tokens in tokenized_texts:
        if word in tokens:
            dfmatrix[i] += 1

idfmatrix = np.log((D + 1) / (dfmatrix + 1)) + 1


tfidf_matrix = tfmatrix * idfmatrix[:, None]
tfidf_matrix = tfidf_matrix.T


LMc = pd.read_csv(LM_PATH)

LMc_dict = {}
for _, row in LMc.iterrows():
    word = row['Word']
    if isinstance(word, str):
        word = word.lower()
        LMc_dict[word] = {
            'positive': row['Positive'],
            'negative': row['Negative'],
            'uncertainty': row['Uncertainty'],
            'constraining': row['Constraining'],
            'litigious': row['Litigious'],
        }


LMc_feature = np.zeros((D, 5))

for j, tokens in enumerate(tokenized_texts):
    for word in tokens:
        if word in LMc_dict:
            LMc_feature[j, 0] += LMc_dict[word]['positive']
            LMc_feature[j, 1] += LMc_dict[word]['negative']
            LMc_feature[j, 2] += LMc_dict[word]['uncertainty']
            LMc_feature[j, 3] += LMc_dict[word]['constraining']
            LMc_feature[j, 4] += LMc_dict[word]['litigious']


Features = np.hstack([tfidf_matrix, LMc_feature])


n_train = len(train_texts)
n_val   = len(val_texts)

X_train = Features[:n_train]
X_val   = Features[n_train:n_train + n_val]
X_test  = Features[n_train + n_val:]


clf = LogisticRegression(max_iter=5000)
clf.fit(X_train, y_train)


y_pred = clf.predict(X_test)


Path("predictions").mkdir(exist_ok=True)

df_out = pd.DataFrame({
    "sentence_id": test_df["sentence_id"],
    "predicted_label": y_pred
})

output_path = "predictions/tfidf_logreg_lm_predictions.csv"
df_out.to_csv(output_path, index=False)

print(f"Saved predictions to {output_path}")


result = evaluate_predictions(output_path)

print(f"\nTest accuracy:  {result['overall']['accuracy']:.4f}")
print(f"Test macro-F1: {result['overall']['macro_f1']:.4f}")

print("\nBy agreement tier:")
for tier, m in result['by_tier'].items():
    print(f"  tier={tier:6s}  acc={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}  (n={m['n']})")

Saved predictions to predictions/logreg_tfidf_predictions.csv

Test accuracy:  0.7521
Test macro-F1: 0.6775

By agreement tier:
  tier=100     acc=0.8717  macro_f1=0.8148  (n=226)
  tier=75-99   acc=0.7500  macro_f1=0.6321  (n=120)
  tier=66-74   acc=0.5526  macro_f1=0.4051  (n=76)
  tier=50-65   acc=0.5645  macro_f1=0.5159  (n=62)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
